In [88]:
import pandas as pd
import numpy as np

crows_pairs = pd.read_csv("data/crows_pairs_anonymized.csv")

In [89]:
counts = crows_pairs["bias_type"].value_counts()
percentages = (counts / counts.sum() * 100).round(2)

taxonomy_overview = (
    pd.DataFrame({
        "count": counts,
        "percentage": percentages
    })
    .reset_index()
    .rename(columns={"index": "bias_type"})
)

taxonomy_overview

,bias_type,count,percentage
0,race-color,516,34.22
1,gender,262,17.37
2,socioeconomic,172,11.41
3,nationality,159,10.54
4,religion,105,6.96
5,age,87,5.77
6,sexual-orientation,84,5.57
7,physical-appearance,63,4.18
8,disability,60,3.98


In [90]:
from collections import defaultdict
from collections import Counter
import re

def tokenize(text):
    return re.findall(r"\b[\w'-]+\b", text.lower())

feature_terms = defaultdict(list)

for _, row in crows_pairs.iterrows():
    words_more = set(tokenize(row["sent_more"]))
    words_less = set(tokenize(row["sent_less"]))

    diff = words_more.symmetric_difference(words_less)

    feature_terms[row["bias_type"]].extend(diff)

for bias_type, terms in feature_terms.items():
    common_terms = Counter(terms).most_common(30)
    print("\n", bias_type)
    print(common_terms)


 race-color
[('white', 258), ('black', 208), ('whites', 39), ('american', 31), ('blacks', 30), ('african', 30), ('americans', 20), ('asian', 19), ('jamal', 15), ('chinese', 15), ('tyrone', 12), ('james', 12), ('people', 11), ('caucasian', 11), ('mexican', 10), ('afro-american', 9), ('jerome', 9), ('native', 8), ('lebron', 8), ('latinos', 7), ('an', 7), ('african-americans', 6), ('shaniqua', 6), ('indians', 6), ('jose', 6), ('john', 5), ('carl', 5), ('tom', 5), ("couldn't", 5), ('a', 5)]

 socioeconomic
[('poor', 79), ('rich', 57), ('white', 21), ('wealthy', 9), ('doctor', 6), ('the', 6), ('ghetto', 5), ('a', 4), ('homeless', 4), ('high', 4), ('poverty', 4), ('uneducated', 4), ('man', 4), ('janitor', 3), ('worker', 3), ('not', 3), ('projects', 3), ('trailer', 3), ('lawyer', 3), ('businessman', 3), ('police', 3), ('meager', 2), ('apartment', 2), ('low', 2), ('stricken', 2), ('community', 2), ('park', 2), ('gated', 2), ('cashier', 2), ('construction', 2)]

 gender
[('he', 75), ('she', 72

In [91]:


def tokenize(text):
    return re.findall(r"\b[\w'-]+\b", text.lower())

terms_more = defaultdict(list)
terms_less = defaultdict(list)

for _, row in crows_pairs.iterrows():
    words_more = set(tokenize(row["sent_more"]))
    words_less = set(tokenize(row["sent_less"]))

    only_more = words_more - words_less
    only_less = words_less - words_more

    bias_type = row["bias_type"]

    terms_more[bias_type].extend(only_more)
    terms_less[bias_type].extend(only_less)

for bias_type in terms_more.keys():
    print("\n" + "="*60)
    print("BIAS TYPE:", bias_type)

    print("\nParole più frequenti SOLO in sent_more:")
    print(Counter(terms_more[bias_type]).most_common(30))

    print("\nParole più frequenti SOLO in sent_less:")
    print(Counter(terms_less[bias_type]).most_common(30))


BIAS TYPE: race-color

Parole più frequenti SOLO in sent_more:
[('black', 194), ('blacks', 29), ('african', 24), ('white', 15), ('jamal', 13), ('asian', 12), ('chinese', 11), ('afro-american', 9), ('american', 9), ('tyrone', 9), ('native', 8), ('mexican', 7), ('latinos', 7), ('americans', 7), ('lebron', 7), ('jerome', 6), ('african-americans', 6), ("couldn't", 5), ('latino', 5), ('mexicans', 5), ('jose', 5), ('indians', 5), ('shaniqua', 4), ('no', 4), ('lee', 4), ('an', 4), ('african-american', 4), ('laquisha', 4), ('asians', 3), ('not', 3)]

Parole più frequenti SOLO in sent_less:
[('white', 243), ('whites', 37), ('american', 22), ('black', 14), ('americans', 13), ('people', 11), ('caucasian', 11), ('james', 10), ('asian', 7), ('african', 6), ('john', 5), ('tom', 5), ('carl', 4), ('chinese', 4), ('william', 4), ('could', 4), ('an', 3), ('spanish', 3), ('jack', 3), ('mexican', 3), ('kate', 3), ('steve', 3), ('tyrone', 3), ('jerome', 3), ('had', 2), ('smith', 2), ('david', 2), ('were',

In [92]:
stop_words = {
    "a", "an", "the", "and", "or", "of", "to", "in", "on", "with",
    "for", "from", "at", "by", "that", "this", "is", "are", "was",
    "were", "be", "been", "being", "have", "has", "had", "do", "does",
    "did", "not", "no", "his", "her", "he", "she", "him",
    "could", "couldn't", "weren't", "haven't", "wasn't",
    "never", "always", "just", "more", "much", "so",
    "but", "as", "t", "all", "their", "people", "don't", "didn't", "hand", "everyone", "two", "less", "begin", "yet", "my", "great", "friends", "similar", "baseball", "projects", "full"

}

manual_map = {
    "blacks": "black",
    "whites": "white",
    "americans": "american",
    "african-americans": "african-american",
    "afro-american": "african-american",
    "asians": "asian",
    "latinos": "latino",
    "mexicans": "mexican",
    "christians": "christian",
    "muslims": "muslim",
    "jews": "jew",
    "heterosexuals": "heterosexual",
    "homosexuals": "homosexual",
    "lesbians": "lesbian",
    "gays": "gay",
    "immigrants": "immigrant",
    "teenagers": "teenager",
    "adults": "adult",
    "children": "child",
    "seniors": "senior",
    "asian-americans": "asian-american",
    "indians": "indian",
    "italians": "italian",
    "foreigners": "foreigner",
    "atheists": "atheist",
    "catholics": "catholic",
    "baptists": "baptist",
    "satanists": "satanist",
    "episcopalians": "episcopalian",
    "straights": "straight",
    "hetrosexual": "heterosexual",
    "hetrosexuals": "heterosexual",
    "etero": "heterosexual",
    "fatties": "fat",
    "retards": "retard"
}

In [93]:
def tokenize_clean(text):
    tokens = re.findall(r"\b[\w'-]+\b", text.lower())

    cleaned = []
    for token in tokens:
        token = manual_map.get(token, token)

        if token in stop_words:
            continue

        cleaned.append(token)

    return cleaned

In [94]:
terms_more = defaultdict(list)
terms_less = defaultdict(list)

for _, row in crows_pairs.iterrows():
    words_more = set(tokenize_clean(row["sent_more"]))
    words_less = set(tokenize_clean(row["sent_less"]))

    only_more = words_more - words_less
    only_less = words_less - words_more

    bias_type = row["bias_type"]

    terms_more[bias_type].extend(only_more)
    terms_less[bias_type].extend(only_less)

for bias_type in terms_more.keys():
    print("\n" + "="*60)
    print("BIAS TYPE:", bias_type)

    print("\nParole più frequenti SOLO in sent_more:")
    print(Counter(terms_more[bias_type]).most_common(30))

    print("\nParole più frequenti SOLO in sent_less:")
    print(Counter(terms_less[bias_type]).most_common(30))


BIAS TYPE: race-color

Parole più frequenti SOLO in sent_more:
[('black', 221), ('african', 24), ('african-american', 19), ('white', 17), ('asian', 15), ('american', 15), ('jamal', 13), ('mexican', 12), ('latino', 12), ('chinese', 11), ('tyrone', 9), ('native', 8), ('indian', 8), ('lebron', 7), ('jerome', 6), ('jose', 5), ('shaniqua', 4), ('lee', 4), ('laquisha', 4), ('hispanic', 3), ('terrance', 3), ('micheal', 3), ('wu', 3), ('chen', 3), ('deshawn', 3), ('asian-american', 3), ('jew', 3), ('chang', 2), ('juan', 2), ('terrell', 2)]

Parole più frequenti SOLO in sent_less:
[('white', 278), ('american', 34), ('black', 15), ('caucasian', 11), ('james', 10), ('asian', 8), ('african', 6), ('john', 5), ('tom', 5), ('carl', 4), ('chinese', 4), ('william', 4), ('spanish', 3), ('jack', 3), ('mexican', 3), ('kate', 3), ('steve', 3), ('tyrone', 3), ('jerome', 3), ('smith', 2), ('david', 2), ('jason', 2), ('juan', 2), ('george', 2), ('left', 2), ('jeff', 2), ('jamal', 2), ('chen', 2), ('caucasian

In [95]:
# unione e stampa leggibile
for bias_type in terms_more.keys():
    print("\n" + "="*70)
    print("BIAS TYPE:", bias_type)

    counter_more = Counter(terms_more[bias_type])
    counter_less = Counter(terms_less[bias_type])

    all_terms = set(counter_more.keys()).union(counter_less.keys())

    rows = []
    for term in all_terms:
        c_more = counter_more[term]
        c_less = counter_less[term]
        c_total = c_more + c_less
        diff = c_more - c_less  # positivo = più in sent_more

        rows.append((term, c_more, c_less,c_total, diff))

    # ordina per differenza assoluta (più discriminanti in alto)
    rows = sorted(rows, key=lambda x: abs(x[3]), reverse=True)

    print(f"{'TERM':20s} {'MORE':>5} {'LESS':>5} {'TOT':>5}{'DIFF':>5}")
    print("-"*50)

    for term, c_more, c_less, c_total, diff in rows[:30]:
        print(f"{term:20s} {c_more:5d} {c_less:5d} {c_total:5d} {diff:5d}")


BIAS TYPE: race-color
TERM                  MORE  LESS   TOT DIFF
--------------------------------------------------
white                   17   278   295  -261
black                  221    15   236   206
american                15    34    49   -19
african                 24     6    30    18
asian                   15     8    23     7
african-american        19     0    19    19
chinese                 11     4    15     7
mexican                 12     3    15     9
jamal                   13     2    15    11
james                    2    10    12    -8
tyrone                   9     3    12     6
latino                  12     0    12    12
caucasian                0    11    11   -11
indian                   8     1     9     7
jerome                   6     3     9     3
lebron                   7     1     8     6
native                   8     0     8     8
shaniqua                 4     2     6     2
jose                     5     1     6     4
chen                     3 

In [96]:
stop_words = {
    "a", "an", "the", "and", "or", "of", "to", "in", "on", "with",
    "for", "from", "at", "by", "that", "this", "is", "are", "was",
    "were", "be", "been", "being", "have", "has", "had", "do", "does",
    "did", "not", "no",
    "could", "couldn't", "weren't", "haven't", "wasn't",
    "never", "always", "just", "more", "much", "so",
    "but", "as", "t", "all", "their", "people", "don't", "didn't",
    "hand", "everyone", "two", "less", "begin", "yet", "my",
    "great", "friends", "similar", "baseball", "projects", "full"
}

bias_specific_noise = {
    "race-color": set(),
    "gender": set(),
    "religion": {"white", "american"},
    "disability": {"white", "man", "smart", "stupid"},
    "sexual-orientation": {"white", "man", "woman", "men", "women", "wife", "husband"},
    "socioeconomic": {"white", "man", "doctor", "lawyer", "police", "businessman"},
    "nationality": {"middle", "new"},
    "physical-appearance": {"yet"},
    "age": {"man", "woman", "men", "boy", "healthy", "great", "similar"}
}

DIRECT_TERMS = {
    "race-color": {
        "black", "white", "asian", "caucasian", "african",
        "african-american", "hispanic", "latino"
    },

    "gender": {
        "he", "she", "him", "her", "his",
        "man", "woman", "men", "women",
        "male", "female", "boy", "girl", "boys", "girls",
        "trans", "cis"
    },

    "religion": {
        "christian", "muslim", "jew", "jewish", "hindu",
        "atheist", "catholic", "buddhist", "baptist",
        "wiccan", "mormon", "lutheran", "methodist",
        "episcopalian"
    },

    "age": {
        "young", "old", "elderly", "teenager", "adult",
        "senior", "child", "teenage", "older",
        "middle-aged", "aged", "minor", "minors"
    },

    "sexual-orientation": {
        "gay", "straight", "heterosexual", "homosexual",
        "lesbian", "bisexual"
    },

    "nationality": {
        "american", "mexican", "italian", "chinese", "indian",
        "russian", "canadian", "japanese", "german", "british",
        "armenian", "arab", "korean", "english", "european"
    },

    "disability": {
        "disabled", "autistic", "wheelchair", "handicapped",
        "dyslexia", "palsy", "syndrome", "mental"
    },

    "physical-appearance": {
        "fat", "overweight", "thin", "skinny", "short", "tall",
        "obese", "slim", "beautiful", "ugly", "muscular",
        "fit", "handsome", "underweight"
    },

    "socioeconomic": {
        "poor", "rich", "wealthy", "homeless",
        "poverty", "low", "low-income", "upper-class"
    }
}

PROXY_TERMS = {
    "race-color": {
        "jamal", "tyrone", "jerome", "shaniqua", "laquisha",
        "jose", "juan", "chen", "lee", "lebron"
    },

    "gender": {
        "mary", "sarah", "jenny", "jennifer", "susan", "susie",
        "marie", "lisa", "judy",
        "tom", "john", "carl", "bill", "jim", "joe", "brad"
    },

    "socioeconomic": {
        "doctor", "lawyer", "businessman",
        "janitor", "cashier", "worker", "laborer",
        "construction", "officer", "cops", "police"
    },

    "nationality": {
        "immigrant", "foreigner", "citizen", "citizens"
    },

    "religion": {
        "church", "mosque", "bible", "torah", "shrine"
    },

    "sexual-orientation": {
        "wife", "husband", "boyfriend", "girlfriend",
        "pride", "openly"
    }
}

STEREOTYPE_TERMS = {
    "socioeconomic": {
        "ghetto", "trailer", "projects", "gang",
        "uneducated", "educated", "successful",
        "posh", "stricken", "meager"
    },

    "disability": {
        "retarded", "retard", "crippled",
        "smart", "stupid", "gifted", "normal"
    },

    "physical-appearance": {
        "monstrous", "disfigured", "hunchback"
    },

    "sexual-orientation": {
        "perverted", "butches"
    }
}

def tokenize_clean(text, bias_type=None):
    tokens = re.findall(r"\b[\w'-]+\b", text.lower())

    cleaned = []
    for token in tokens:
        token = manual_map.get(token, token)

        if token in stop_words:
            continue

        if bias_type is not None and token in bias_specific_noise.get(bias_type, set()):
            continue

        cleaned.append(token)

    return cleaned

def classify_term(term, bias_type):
    if term in DIRECT_TERMS.get(bias_type, set()):
        return "direct"

    if term in PROXY_TERMS.get(bias_type, set()):
        return "proxy"

    if term in STEREOTYPE_TERMS.get(bias_type, set()):
        return "stereotype"

    return "other"

In [97]:
terms_more = defaultdict(list)
terms_less = defaultdict(list)

for _, row in crows_pairs.iterrows():
    bias_type = row["bias_type"]

    words_more = set(tokenize_clean(row["sent_more"], bias_type))
    words_less = set(tokenize_clean(row["sent_less"], bias_type))

    only_more = words_more - words_less
    only_less = words_less - words_more

    terms_more[bias_type].extend(only_more)
    terms_less[bias_type].extend(only_less)

In [98]:
for bias_type in terms_more.keys():
    print("\n" + "="*70)
    print("BIAS TYPE:", bias_type)

    counter_more = Counter(terms_more[bias_type])
    counter_less = Counter(terms_less[bias_type])

    all_terms = set(counter_more.keys()).union(counter_less.keys())

    rows = []
    for term in all_terms:
        c_more = counter_more[term]
        c_less = counter_less[term]
        c_total = c_more + c_less
        diff = c_more - c_less

        term_type = classify_term(term, bias_type)

        rows.append((term, c_more, c_less, c_total, diff, term_type))

    rows = sorted(rows, key=lambda x: (x[3], abs(x[4])), reverse=True)

    print(f"{'TERM':20s} {'MORE':>5} {'LESS':>5} {'TOT':>5} {'DIFF':>6} {'TYPE':>12}")
    print("-" * 65)

    for term, c_more, c_less, c_total, diff, term_type in rows[:30]:
        print(f"{term:20s} {c_more:5d} {c_less:5d} {c_total:5d} {diff:6d} {term_type:>12}")

    for target_type in ["direct", "proxy", "stereotype"]:
        print(f"\n--- {target_type.upper()} TERMS ---")

        filtered_rows = [
            row for row in rows
            if row[5] == target_type
        ]

        for term, c_more, c_less, c_total, diff, term_type in filtered_rows[:20]:
            print(f"{term:20s} {c_more:5d} {c_less:5d} {c_total:5d} {diff:6d}")


BIAS TYPE: race-color
TERM                  MORE  LESS   TOT   DIFF         TYPE
-----------------------------------------------------------------
white                   17   278   295   -261       direct
black                  221    15   236    206       direct
american                15    34    49    -19        other
african                 24     6    30     18       direct
asian                   15     8    23      7       direct
african-american        19     0    19     19       direct
jamal                   13     2    15     11        proxy
mexican                 12     3    15      9        other
chinese                 11     4    15      7        other
latino                  12     0    12     12       direct
james                    2    10    12     -8        other
tyrone                   9     3    12      6        proxy
caucasian                0    11    11    -11       direct
indian                   8     1     9      7        other
jerome                   6

In [99]:
import json

bias_lexicon = {
    "source_dataset": "CrowS-Pairs",
    "version": "v1",
    "features": {}
}

for bias_type in terms_more.keys():
    counter_more = Counter(terms_more[bias_type])
    counter_less = Counter(terms_less[bias_type])
    all_terms = set(counter_more.keys()).union(counter_less.keys())

    bias_lexicon["features"][bias_type] = {
        "direct_terms": [],
        "proxy_terms": [],
        "stereotype_terms": [],
        "other_terms": []
    }

    for term in all_terms:
        term_type = classify_term(term, bias_type)

        entry = {
            "term": term,
            "more": counter_more[term],
            "less": counter_less[term],
            "total": counter_more[term] + counter_less[term],
            "diff": counter_more[term] - counter_less[term]
        }

        if term_type == "direct":
            bias_lexicon["features"][bias_type]["direct_terms"].append(entry)
        elif term_type == "proxy":
            bias_lexicon["features"][bias_type]["proxy_terms"].append(entry)
        elif term_type == "stereotype":
            bias_lexicon["features"][bias_type]["stereotype_terms"].append(entry)
        else:
            bias_lexicon["features"][bias_type]["other_terms"].append(entry)

with open("bias_lexicon_crows_pairs.json", "w") as f:
    json.dump(bias_lexicon, f, indent=2)

In [100]:
import json

bias_lexicon_light = {
    "source_dataset": "CrowS-Pairs",
    "features": {}
}

for bias_type in terms_more.keys():
    counter_more = Counter(terms_more[bias_type])
    counter_less = Counter(terms_less[bias_type])

    all_terms = set(counter_more.keys()).union(counter_less.keys())

    bias_lexicon_light["features"][bias_type] = {
        "direct": [],
        "proxy": [],
        "stereotype": []
    }

    for term in all_terms:
        term_type = classify_term(term, bias_type)

        if term_type == "direct":
            bias_lexicon_light["features"][bias_type]["direct"].append(term)
        elif term_type == "proxy":
            bias_lexicon_light["features"][bias_type]["proxy"].append(term)
        elif term_type == "stereotype":
            bias_lexicon_light["features"][bias_type]["stereotype"].append(term)

# (opzionale) ordina alfabeticamente
for bias_type in bias_lexicon_light["features"]:
    for t in ["direct", "proxy", "stereotype"]:
        bias_lexicon_light["features"][bias_type][t] = sorted(
            list(set(bias_lexicon_light["features"][bias_type][t]))
        )

with open("bias_lexicon_crows_pairs_light.json", "w") as f:
    json.dump(bias_lexicon_light, f, indent=2)

# Analisi di CrowS-Pairs per la costruzione della tassonomia delle feature sensibili

In questa sezione viene presentato il processo completo di analisi del dataset **CrowS-Pairs**, con l’obiettivo di estrarre in modo sistematico e data-driven le feature sensibili e i termini associati, da utilizzare successivamente nella generazione di test controfattuali per la valutazione della fairness.

---

## Obiettivo

L’obiettivo principale di questa fase è costruire una **tassonomia operativa delle feature sensibili**, non definita manualmente a priori, ma derivata direttamente dai dati.
In particolare, vogliamo:

- identificare le principali categorie di bias presenti nel dataset;
- estrarre i termini che rappresentano tali categorie;
- pulire e normalizzare i termini;
- classificarli in base al loro ruolo semantico (diretti, proxy, stereotipi).

---

## Struttura del dataset

Il dataset CrowS-Pairs è composto da coppie di frasi:

- `sent_more`: frase maggiormente stereotipata;
- `sent_less`: frase meno stereotipata;
- `bias_type`: categoria di bias associata alla coppia.

L’idea chiave è che la differenza tra le due frasi sia guidata da una **feature sensibile** (ad esempio genere, etnia, religione, ecc.).

---

## Analisi della distribuzione dei bias

Come primo passo, viene analizzata la distribuzione delle categorie di bias presenti nel dataset.
Questo consente di ottenere una prima fotografia della tassonomia:

```text
race-color, gender, socioeconomic, nationality,
religion, age, sexual-orientation, physical-appearance, disability
```

Questa distribuzione fornisce una base empirica per selezionare le categorie più rilevanti nel contesto del progetto.

* * *

## Estrazione dei termini candidati

Per ogni coppia di frasi, vengono estratti i termini che differiscono tra `sent_more` e `sent_less`.
L’idea è che questi termini rappresentino:

-   valori della feature sensibile;
-   proxy della feature sensibile;
-   oppure elementi stereotipati.

Formalmente, per ogni coppia:

-   si tokenizzano le due frasi;
-   si calcola la differenza tra gli insiemi di parole.

Questo produce una lista di termini candidati per ogni categoria di bias.

## Separazione tra contesto stereotipato e non

Per migliorare l’analisi, i termini vengono separati in:

-   termini presenti solo in `sent_more`;
-   termini presenti solo in `sent_less`.

Questo permette di osservare **da quale lato della coppia compare un termine**, fornendo un’indicazione sulla sua associazione con contenuti stereotipati.

* * *

## Pulizia e normalizzazione

L’estrazione automatica introduce molto rumore (articoli, preposizioni, parole generiche).
Per questo motivo viene applicata una fase di cleaning composta da:

### 1\. Stop words

Viene definita una lista di parole non informative da rimuovere (es. _the, a, is, and, people_).

### 2\. Normalizzazione manuale

Viene introdotta una mappatura per uniformare varianti morfologiche:
- `blacks → black`
- `whites → white`
- `christians → christian`
- `muslims → muslim`
- `teenagers → teenager`

### 3\. Rumore specifico per categoria

Alcuni termini sono rilevanti in una categoria ma rumore in un’altra.
Ad esempio:

-   `white` è rilevante per _race-color_;
-   ma può essere rumore in _religion_ o _disability_.

Per questo motivo viene introdotto un filtro specifico per ciascun `bias_type`.

## Costruzione delle statistiche

Per ogni termine vengono calcolate:

-   `MORE`: frequenza nel contesto stereotipato;
-   `LESS`: frequenza nel contesto non stereotipato;
-   `TOT`: frequenza totale;
-   `DIFF`: differenza tra le due.

Queste metriche permettono di identificare i termini più rilevanti e discriminanti.

* * *

## Classificazione semantica dei termini

I termini estratti vengono classificati in quattro categorie:

### 1\. Direct

Termini che rappresentano esplicitamente la feature sensibile.

Esempi:

`black, white, woman, muslim, young, gay`

### 2\. Proxy
Termini che suggeriscono indirettamente la feature sensibile.
Esempi:

```text
Jamal, Tyrone → race
Mary, Tom → gender
church, mosque → religion
```

### 3\. Stereotype
Termini associati a stereotipi o giudizi.
Esempi:

`ghetto, uneducated, perverted, ugly`

### 4\. Other

Termini non rilevanti o ambigui.

## Risultato dell’analisi


Il risultato finale è una struttura organizzata del tipo:

```bash
bias\_type
  ├── direct terms
  ├── proxy terms
  ├── stereotype terms
  └── other
```

Questa rappresentazione consente di passare da un dataset non strutturato a una **tassonomia esplicita delle feature sensibili**.


## Implicazioni per la fairness

Questa analisi è fondamentale perché:

1.  evita di definire manualmente le feature sensibili;
2.  permette di identificare quali termini sono realmente utilizzati nei benchmark;
3.  distingue tra:
    -   feature sensibili (usabili per controfattuali),
    -   proxy (usabili per test avanzati),
    -   stereotipi (usabili per analisi ma non per generazione).

* * *

## Uso nella generazione controfattuale

I termini classificati come **direct** rappresentano i candidati principali per la generazione di test controfattuali.

Esempi:

```text
gender → he/she, man/woman
race-color → black/white, asian/white
religion → christian/muslim
age → young/old
sexual-orientation → gay/straight
nationality → american/mexican
```

Questi termini permettono di creare input in cui si modifica solo la feature sensibile, mantenendo invariato il resto del contesto, in linea con il principio di counterfactual fairness.

## Conclusione


Attraverso questa pipeline, il dataset CrowS-Pairs viene trasformato in uno strumento operativo per:

-   costruire una tassonomia delle feature sensibili;
-   identificare termini rilevanti per ciascuna categoria;
-   supportare la generazione automatica di test controfattuali.

Questo passaggio rappresenta il collegamento tra l’analisi dei benchmark esistenti e la costruzione della pipeline di fairness proposta nel progetto.